# IFRS 9, IRB capital, and scenario stress

Keep accounting ECL and prudential capital distinct while reconciling shared PD, LGD, and EAD inputs.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
import numpy as np
import pandas as pd

from creditriskbook.capital import corporate_irb_capital
from creditriskbook.ecl import educational_ecl

portfolio = pd.DataFrame({
    "stage": [1, 1, 2, 2, 3],
    "pd_12m": [0.005, 0.02, 0.04, 0.10, 0.45],
    "lgd": [0.35, 0.40, 0.45, 0.55, 0.65],
    "ead": [1_000_000, 750_000, 500_000, 300_000, 100_000],
    "remaining_months": [12, 36, 48, 24, 18],
    "effective_interest_rate": [0.04, 0.05, 0.045, 0.06, 0.07],
})
ecl = educational_ecl(portfolio)
assert (ecl["ecl_downside"] >= ecl["ecl_base"]).all()
print(ecl[["stage", "ecl_upside", "ecl_base", "ecl_downside", "ecl_probability_weighted"]])

In [ ]:
irb = corporate_irb_capital(
    portfolio["pd_12m"].to_numpy(), portfolio["lgd"].to_numpy(), portfolio["ead"].to_numpy(),
    maturity_years=np.clip(portfolio["remaining_months"].to_numpy() / 12, 1, 5),
)
reconciliation = pd.DataFrame({
    "ead": portfolio["ead"],
    "ifrs9_ecl": ecl["ecl_probability_weighted"],
    "irb_expected_loss": irb["expected_loss"],
    "irb_capital": irb["capital"],
    "rwa": irb["risk_weighted_assets"],
})
print(reconciliation)
assert np.allclose(reconciliation["rwa"], 12.5 * reconciliation["irb_capital"])

Do not force equality between IFRS 9 ECL and IRB expected loss. Horizon, cycle philosophy, floors, downturn concepts, scenario treatment, discounting, and regulatory scope differ.